# Python Project
## T-bank flights and hotels dataset

In [26]:
import pandas as pd
import plotly as pt

with open("dano_dataset_travel.csv",encoding="utf-8") as file:
    print(file.readline())

df = pd.read_csv("dano_dataset_travel.csv", sep=";",decimal=",",engine='pyarrow')

print(f"The original size: {df.shape[0]:,} rows, {df.shape[1]} cols")

order_online_payment_flg;account_rk;client_rk;order_rk;loyalty_program_type_nm;bundle_nm;order_type_cd;order_status_cd;party_first_order_dt;party_first_order_type_dt;free_cancel_booking_dttm;created_dttm;cancel_dttm;book_start_dttm;local_book_start_dttm;book_end_dttm;hotel_country;hotel_city;avia_dep_city;avia_arr_city;promo_code_discount_amt;loyalty_accrual_rub_amt;nominal_price_eur_amt;nominal_price_rub_amt;order_item_cnt;month_beginning_balance_rub;monthly_income_amt;suppress_email_flg;suppress_call_flg;bounce_cd;last_sms_success_flg;call_contact_6m_flg;call_contact_3m_flg;call_contact_1m_flg;good_email_address_flg;bad_email_address_flg;email_valid_flg;children_cnt;age;age_type_cd;parent_meeting_region_nm;delivery_region_category_cd;lvn_city_nm;lvn_state_nm;time_zone_delta_tm;time_zone_cd;last_used_product_cd;first_used_product_cd;mobile_phone_operator_nm;marital_status_cd;education_level_cd;birth_place;gender_cd;last_sms_dt;last_email_send_dt;last_session_dttm

The original size: 8

## Step 0: Description of the dataset

In [33]:
import pandas as pd

df = pd.read_csv('dano_dataset_travel.csv', sep=';', low_memory=False)

# Конвертируем нужные колонки из строк в числа
numeric_cols = [
    'nominal_price_rub_amt',
    'monthly_income_amt',
    'age',
    'loyalty_accrual_rub_amt',
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.', regex=False),
        errors='coerce'
    )

# Описательная статистика
df[numeric_cols].describe()

,nominal_price_rub_amt,monthly_income_amt,age,loyalty_accrual_rub_amt
count,7.868850e+05,6.730780e+05,835937.000000,475353.000000
mean,1.517810e+04,1.473086e+05,36.073155,644.108014
std,2.402879e+04,2.660834e+05,9.370449,1586.753948
min,1.000000e+00,0.000000e+00,14.000000,-8645.000000
25%,5.090000e+03,5.360000e+04,29.000000,154.000000
50%,9.342000e+03,9.715000e+04,35.000000,326.000000
75%,1.652700e+04,1.675000e+05,41.000000,682.000000
max,2.025808e+06,6.700000e+07,98.000000,606732.000000


In [34]:
import pandas as pd
import numpy as np

# ============================================================
# Загрузка сырого датасета (до очистки)
# ============================================================

df = pd.read_csv('dano_dataset_travel.csv', sep=';', low_memory=False)

# Числа в датасете записаны с запятой вместо точки ('142,0' → 142.0).
# Конвертируем только для подсчёта статистики — датасет не изменяем.
numeric_cols = [
    'nominal_price_rub_amt',    # Сумма заказа в рублях
    'monthly_income_amt',       # Месячный доход клиента
    'age',                      # Возраст клиента
    'loyalty_accrual_rub_amt',  # Начисленные баллы лояльности (руб.)
]

df_num = df[numeric_cols].copy()
for col in numeric_cols:
    df_num[col] = pd.to_numeric(
        df_num[col].astype(str).str.replace(',', '.', regex=False),
        errors='coerce'
    )


# ============================================================
# Описательная статистика по 4 числовым полям
# ============================================================

print("=" * 60)
print("ОПИСАТЕЛЬНАЯ СТАТИСТИКА — ИСХОДНЫЙ ДАТАСЕТ")
print(f"Всего строк в датасете: {len(df):,}")
print("=" * 60)

fields_info = {
    'nominal_price_rub_amt': 'Сумма заказа (руб.)',
    'monthly_income_amt':    'Месячный доход (руб.)',
    'age':                   'Возраст клиента (лет)',
    'loyalty_accrual_rub_amt': 'Баллы лояльности (руб.)',
}

for col, label in fields_info.items():
    s = df_num[col]
    print(f"\n--- {label} [{col}] ---")
    print(f"  Непустых значений : {s.count():>12,.0f}  "
          f"(пропусков: {s.isna().sum():,} — {s.isna().mean()*100:.1f}%)")
    print(f"  Среднее (mean)    : {s.mean():>12,.2f}")
    print(f"  Медиана (median)  : {s.median():>12,.2f}")
    print(f"  Ст. отклонение    : {s.std():>12,.2f}")
    print(f"  Минимум           : {s.min():>12,.2f}")
    print(f"  25-й перцентиль   : {s.quantile(0.25):>12,.2f}")
    print(f"  75-й перцентиль   : {s.quantile(0.75):>12,.2f}")
    print(f"  Максимум          : {s.max():>12,.2f}")


# ============================================================
# Сводная таблица одним вызовом (удобно для отчёта)
# ============================================================

print("\n\n" + "=" * 60)
print("СВОДНАЯ ТАБЛИЦА")
print("=" * 60)

summary = df_num.agg(['count', 'mean', 'median', 'std', 'min', 'max'])
summary.loc['missing_%'] = (df_num.isna().mean() * 100).round(1)
summary.columns = [fields_info[c] for c in summary.columns]
print(summary.round(2).to_string())

ОПИСАТЕЛЬНАЯ СТАТИСТИКА — ИСХОДНЫЙ ДАТАСЕТ
Всего строк в датасете: 835,938

--- Сумма заказа (руб.) [nominal_price_rub_amt] ---
  Непустых значений :      786,885  (пропусков: 49,053 — 5.9%)
  Среднее (mean)    :    15,178.10
  Медиана (median)  :     9,342.00
  Ст. отклонение    :    24,028.79
  Минимум           :         1.00
  25-й перцентиль   :     5,090.00
  75-й перцентиль   :    16,527.00
  Максимум          : 2,025,808.00

--- Месячный доход (руб.) [monthly_income_amt] ---
  Непустых значений :      673,078  (пропусков: 162,860 — 19.5%)
  Среднее (mean)    :   147,308.63
  Медиана (median)  :    97,150.00
  Ст. отклонение    :   266,083.38
  Минимум           :         0.00
  25-й перцентиль   :    53,600.00
  75-й перцентиль   :   167,500.00
  Максимум          : 67,000,000.00

--- Возраст клиента (лет) [age] ---
  Непустых значений :      835,937  (пропусков: 1 — 0.0%)
  Среднее (mean)    :        36.07
  Медиана (median)  :        35.00
  Ст. отклонение    :         9.37
 

## Step 1: Removing inconsistent data

### Columns

1. Identifier columns: just hashes, have no analytical value

In [19]:
id_cols = ['account_rk', 'client_rk', 'order_rk']

2. Almost empty columns (>90% rows are empty)

In [20]:
mostly_empty_cols = [
    'cancel_dttm',              # 100% empty
    'call_contact_6m_flg',      # 100 empty
    'call_contact_3m_flg',      # 100% empty
    'call_contact_1m_flg',      # 100% empty
    'bounce_cd',                # 95% empty
    'free_cancel_booking_dttm', # 91% empty
    'good_email_address_flg',   # 98% empty
    'bad_email_address_flg',    # 98% empty
    'last_email_send_dt',       # 98% empty
]

3. Other Columns

In [ ]:
duplicate_cols = ['local_book_start_dttm'] # the same as the "book_start_dttm" column, but in the local time zone.
# No sense to keep them both

noisy_cols = ['birth_place'] # Place of birth: the text is too loose (city, region, country — all mixed up),
# difficult to use in analysis

In [23]:
cols_to_drop = id_cols + mostly_empty_cols + duplicate_cols + noisy_cols
df.drop(columns=cols_to_drop, inplace=True)

In [16]:
print(f"After deleting: {df.shape[1]} columns left")

After deleting: 42 columns left


### Cleaning inconsistent rows

In [ ]:
# More than 49000 rows have no information in the field order_type_cd, 
before = len(df)
df = df[df['order_type_cd'].notna()].copy()
print(f"The number of rows without an order deleted: {before - len(df):,}")

The number of rows without an order deleted: 0
